In [1]:
import re
import time
import pandas as pd

from bs4 import BeautifulSoup
from datetime import datetime
from zoneinfo import ZoneInfo

from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# ============================================================
# SETTINGS
# ============================================================

BASE_URL = "https://fuelo.net/prices/last_updated"
OUTPUT_CSV = "fuelo_all_prices.csv"

# Fuelo pagination uses offsets: /page/20, /page/40, ...
PAGE_STEP = 20

# Safety ceiling. The scraper stops earlier when pages become empty.
MAX_OFFSET = 10000

# Stop after this many consecutive pages without station cards.
EMPTY_PAGE_LIMIT = 3

# Delay between page loads. Increase if Fuelo starts throttling.
PAGE_DELAY_SECONDS = 1.0
INITIAL_LOAD_SECONDS = 3.0

HEADLESS = True


# ============================================================
# REGION TO CITIES MAPPING
# ============================================================

REGION_CITIES = {
    "Благоевград": [
        "Благоевград", "Петрич", "Сандански", "Гоце Делчев", "Разлог",
        "Банско", "Симитли", "Кресна", "Хаджидимово", "Якоруда", "Кулата"
    ],
    "Бургас": [
        "Бургас", "Айтос", "Карнобат", "Поморие", "Несебър", "Созопол",
        "Средец", "Царево", "Камено", "Приморско", "Малко Търново", "Обзор"
    ],
    "Варна": [
        "Варна", "Провадия", "Девня", "Аксаково", "Белослав", "Дългопол",
        "Долни чифлик", "Суворово", "Вълчи дол"
    ],
    "Велико Търново": [
        "Велико Търново", "Горна Оряховица", "Свищов", "Павликени",
        "Полски Тръмбеш", "Елена", "Лясковец", "Златарица", "Стражица", "Дебелец"
    ],
    "Видин": [
        "Видин", "Белоградчик", "Дунавци", "Кула", "Димово",
        "Грамада", "Брегово", "Ружинци", "Макреш", "Ново село"
    ],
    "Враца": [
        "Враца", "Козлодуй", "Мездра", "Бяла Слатина", "Оряхово",
        "Мизия", "Криводол", "Роман", "Хайредин", "Борован"
    ],
    "Габрово": [
        "Габрово", "Севлиево", "Дряново", "Трявна", "Плачковци",
        "Градница", "Добромирка", "Шумата", "Кръвеник", "Стоките"
    ],
    "Добрич": [
        "Добрич", "Балчик", "Генерал Тошево", "Каварна", "Тервел",
        "Шабла", "Крушари", "Албена", "Оброчище", "Кранево"
    ],
    "Кърджали": [
        "Кърджали", "Момчилград", "Крумовград", "Ардино", "Джебел",
        "Кирково", "Черноочене", "Перперек", "Бенковски", "Фотиново"
    ],
    "Кюстендил": [
        "Кюстендил", "Дупница", "Бобов дол", "Сапарева баня", "Рила",
        "Кочериново", "Бобошево", "Невестино", "Трекляно", "Ресилово"
    ],
    "Ловеч": [
        "Ловеч", "Троян", "Тетевен", "Луковит", "Априлци",
        "Угърчин", "Ябланица", "Летница", "Орешак", "Гложене"
    ],
    "Монтана": [
        "Монтана", "Лом", "Берковица", "Вършец", "Вълчедръм",
        "Бойчиновци", "Чипровци", "Медковец", "Якимово", "Брусарци"
    ],
    "Пазарджик": [
        "Пазарджик", "Велинград", "Пещера", "Панагюрище", "Ракитово",
        "Септември", "Белово", "Брацигово", "Стрелча", "Батак"
    ],
    "Перник": [
        "Перник", "Радомир", "Брезник", "Трън", "Земен",
        "Ковачевци", "Батановци", "Драгичево", "Рударци", "Дивотино"
    ],
    "Плевен": [
        "Плевен", "Червен бряг", "Кнежа", "Левски", "Белене",
        "Долна Митрополия", "Пордим", "Гулянци", "Искър", "Славяново"
    ],
    "Пловдив": [
        "Пловдив", "Асеновград", "Карлово", "Раковски", "Първомай",
        "Хисаря", "Сопот", "Стамболийски", "Куклен", "Радиново"
    ],
    "Разград": [
        "Разград", "Исперих", "Кубрат", "Лозница", "Цар Калоян",
        "Завет", "Самуил", "Ясеновец", "Гецово", "Сеново"
    ],
    "Русе": [
        "Русе", "Бяла", "Ветово", "Две могили", "Борово",
        "Сливо поле", "Иваново", "Ценово", "Мартен", "Николово"
    ],
    "Силистра": [
        "Силистра", "Тутракан", "Дулово", "Главиница", "Алфатар",
        "Ситово", "Кайнарджа", "Айдемир", "Калипетрово", "Сребърна"
    ],
    "Сливен": [
        "Сливен", "Нова Загора", "Котел", "Твърдица", "Шивачево",
        "Кермен", "Желю войвода", "Градец", "Сотиря"
    ],
    "Смолян": [
        "Смолян", "Златоград", "Мадан", "Рудозем", "Девин",
        "Чепеларе", "Неделино", "Доспат", "Борино", "Пампорово"
    ],
    "София": [
        "София"
    ],
    "София област": [
        "Ботевград", "Самоков", "Своге", "Елин Пелин", "Костинброд",
        "Ихтиман", "Пирдоп", "Сливница", "Правец", "Копривщица"
    ],
    "Стара Загора": [
        "Стара Загора", "Казанлък", "Чирпан", "Раднево", "Гълъбово",
        "Мъглиж", "Гурково", "Николаево", "Шипка", "Павел баня"
    ],
    "Търговище": [
        "Търговище", "Попово", "Омуртаг", "Антоново", "Опака",
        "Стража", "Дралфа", "Подгорица", "Макариополско"
    ],
    "Хасково": [
        "Хасково", "Димитровград", "Свиленград", "Харманли", "Любимец",
        "Ивайловград", "Симеоновград", "Тополовград", "Минерални бани", "Меричлери"
    ],
    "Шумен": [
        "Шумен", "Нови пазар", "Велики Преслав", "Смядово", "Каспичан",
        "Каолиново", "Плиска", "Върбица", "Хитрино", "Венец"
    ],
    "Ямбол": [
        "Ямбол", "Елхово", "Стралджа", "Болярово", "Кукорево",
        "Веселиново", "Безмер", "Калчево", "Роза"
    ],
}

CITY_TO_REGION = {
    city: region
    for region, cities in REGION_CITIES.items()
    for city in cities
}


# ============================================================
# CONSTANTS
# ============================================================

COLUMNS = [
    "created_at",
    "city",
    "station",
    "fuel",
    "price",
    "region",
    "location",
]

# Canonical chain/brand names. Matching is case-insensitive and prefix-based.
STATION_BRANDS = [
    ("omv", "OMV"),
    ("shell", "Shell"),
    ("lukoil", "Lukoil"),
    ("лукойл", "Lukoil"),
    ("eko", "ЕКО"),
    ("еко", "ЕКО"),
    ("rompetrol", "Rompetrol"),
    ("ромпетрол", "Rompetrol"),
    ("petrol", "Petrol"),
    ("петрол", "Petrol"),
    ("avia", "AVIA"),
    ("инса ойл", "Insa Oil"),
    ("insa oil", "Insa Oil"),
    ("gazprom", "Gazprom"),
    ("газпром", "Gazprom"),
    ("cruise", "Cruise"),
    ("круиз", "Cruise"),
    ("dieselor", "Dieselor"),
    ("зара", "Зара"),
    ("топливо", "Топливо"),
]


# ============================================================
# HELPERS
# ============================================================

def normalize_spaces(value):
    if value is None:
        return None
    return re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip()


def get_region_by_city(city):
    """
    Keeps the original mapping, but DOES NOT discard a station when a city
    is missing from the mapping. This is important when scraping all Fuelo
    stations, including small towns and villages.
    """
    city = normalize_spaces(city)

    if not city:
        return None

    region = CITY_TO_REGION.get(city)

    if not region:
        print(f"[REGION] Missing mapping for city: {city}")

    return region


def create_driver():
    options = Options()

    if HEADLESS:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(45)

    return driver


def get_soup(driver, url, first_page=False):
    driver.get(url)
    time.sleep(INITIAL_LOAD_SECONDS if first_page else PAGE_DELAY_SECONDS)

    html = driver.page_source

    print(f"[PAGE] {url} | HTML: {len(html):,} chars")

    return BeautifulSoup(html, "html.parser")


def make_page_url(offset):
    if offset == 0:
        return f"{BASE_URL}?lang=bg"

    return f"{BASE_URL}/page/{offset}?lang=bg"


def clean_price(price_text):
    """
    Extract the first decimal-looking price from text such as:
      2,41 лв./л
      1.23 €/л
      2,33 лв./кг
    """
    price_text = normalize_spaces(price_text)

    if not price_text:
        raise ValueError("Empty price")

    match = re.search(r"(?<!\d)(\d{1,3}(?:[.,]\d{1,3})?)(?!\d)", price_text)

    if not match:
        raise ValueError(f"No numeric price found in: {price_text}")

    return float(match.group(1).replace(",", "."))


def normalize_fuel_name(product):
    """
    Normalize products from different Fuelo brands into stable categories.

    The raw brand-specific product name is intentionally mapped to a common
    category so the output remains compatible with the existing CSV/database.
    """
    raw = normalize_spaces(product)

    if not raw:
        return None

    p = raw.casefold()

    # Skip electricity / EV charging.
    if any(x in p for x in ["електр", "electric", "kwh", "kw"]):
        return None

    # LPG / propane-butane.
    if any(x in p for x in [
        "lpg", "autogas", "автогаз", "пропан", "бутан",
        "blue force gas",
    ]):
        return "Пропан Бутан"

    # CNG / methane.
    if any(x in p for x in ["метан", "methane", "cng"]):
        return "Метан"

    # Diesel.
    if any(x in p for x in ["diesel", "дизел", "нафта"]):
        premium_tokens = [
            "premium", "премиум", "maxxmotion", "ecto",
            "v-power", "double filtered", "green force",
            "diesel+", "diesel +", "topdiesel", "топдизел",
        ]

        if any(x in p for x in premium_tokens):
            return "Дизел премиум"

        return "Дизел"

    # Gasoline 100.
    if re.search(r"(?<!\d)100(?!\d)", p):
        return "Бензин A100"

    # Gasoline 98.
    if re.search(r"(?<!\d)98(?!\d)", p):
        return "Бензин A98"

    # Gasoline 95.
    if (
        re.search(r"(?<!\d)95(?!\d)", p)
        or "a95" in p
        or "а95" in p
        or "a-95" in p
        or "а-95" in p
        or "super 95" in p
    ):
        return "Бензин A95"

    # Other gasoline descriptions where the octane cannot be identified.
    if any(x in p for x in ["бензин", "gasoline", "petrol"]):
        return "Бензин"

    print(f"[FUEL] Unmapped product: {raw}")
    return None


def parse_created_at(card_text):
    """
    Convert Fuelo's displayed update timestamp to UTC ISO 8601.
    Falls back to scrape time when Fuelo does not expose an absolute timestamp.
    """
    card_text = normalize_spaces(card_text) or ""
    now_sofia = datetime.now(ZoneInfo("Europe/Sofia"))

    # Example: последно обновяване 28.08.2026 18:30
    absolute = re.search(
        r"(?:последно\s+обновяване|обновено|актуализирано)?\s*"
        r"(\d{1,2}\.\d{1,2}\.\d{4})\s+(\d{1,2}:\d{2})",
        card_text,
        flags=re.IGNORECASE,
    )

    if absolute:
        local_dt = datetime.strptime(
            f"{absolute.group(1)} {absolute.group(2)}",
            "%d.%m.%Y %H:%M",
        ).replace(tzinfo=ZoneInfo("Europe/Sofia"))

        return local_dt.astimezone(ZoneInfo("UTC")).isoformat()

    # Example: Днес 18:30
    today_match = re.search(
        r"днес\s+(?:в\s+)?(\d{1,2}:\d{2})",
        card_text,
        flags=re.IGNORECASE,
    )

    if today_match:
        hour, minute = map(int, today_match.group(1).split(":"))
        local_dt = now_sofia.replace(
            hour=hour,
            minute=minute,
            second=0,
            microsecond=0,
        )
        return local_dt.astimezone(ZoneInfo("UTC")).isoformat()

    return now_sofia.astimezone(ZoneInfo("UTC")).isoformat()


def extract_station_name(card):
    h1_link = card.select_one("h1 a")

    if not h1_link:
        return None

    return normalize_spaces(h1_link.get_text(" ", strip=True))


def normalize_station_name(station_name, city=None):
    """
    Store a canonical brand when it is recognized.
    For independent stations, keep the Fuelo station name instead of dropping it.
    """
    station_name = normalize_spaces(station_name)

    if not station_name:
        return None

    normalized = station_name.casefold()

    for prefix, canonical in STATION_BRANDS:
        if normalized.startswith(prefix.casefold()):
            return canonical

    # Remove generic leading word when Fuelo labels an independent station as
    # "Бензиностанция XYZ".
    fallback = re.sub(
        r"^бензиностанция\s+",
        "",
        station_name,
        flags=re.IGNORECASE,
    ).strip()

    # If the station title ends with the city, remove only that suffix.
    city = normalize_spaces(city)
    if city:
        fallback = re.sub(
            rf"\s*[-,:]?\s*{re.escape(city)}\s*$",
            "",
            fallback,
            flags=re.IGNORECASE,
        ).strip()

    return fallback or station_name


def extract_city_and_location(card):
    h4 = card.select_one("h4")

    if not h4:
        return None, None

    city_element = h4.select_one("a")

    if not city_element:
        return None, normalize_spaces(h4.get_text(" ", strip=True))

    city = normalize_spaces(city_element.get_text(" ", strip=True))
    full_location = normalize_spaces(h4.get_text(" ", strip=True))

    location = full_location

    if city and location:
        # Remove only the first occurrence of the city.
        location = re.sub(
            rf"^\s*{re.escape(city)}\s*[,\-:]?\s*",
            "",
            location,
            count=1,
            flags=re.IGNORECASE,
        )
        location = location.replace('"', "").strip(" ,")

    return city, location


def find_station_cards(soup):
    """
    Preserve the selectors proven by the original scraper, but centralize the
    card detection so it is easy to adjust if Fuelo changes its markup.
    """
    cards = []

    for card in soup.select("div.row"):
        if (
            card.select_one("h1 a")
            and card.select_one("h4")
            and card.select_one("table.table")
        ):
            cards.append(card)

    return cards


def scrape_station_card(card):
    results = []

    station_title = extract_station_name(card)

    if not station_title:
        return results

    city, location = extract_city_and_location(card)
    region = get_region_by_city(city)
    station = normalize_station_name(station_title, city=city)

    card_text = card.get_text(" ", strip=True)
    created_at = parse_created_at(card_text)

    table = card.select_one("table.table")

    if not table:
        return results

    for row in table.select("tbody tr"):
        cells = row.select("td")

        if len(cells) < 3:
            continue

        product_raw = normalize_spaces(cells[1].get_text(" ", strip=True))
        price_raw = normalize_spaces(cells[2].get_text(" ", strip=True))

        if not product_raw or not price_raw:
            continue

        fuel = normalize_fuel_name(product_raw)

        if not fuel:
            continue

        try:
            price = clean_price(price_raw)
        except ValueError as exc:
            print(f"[PRICE] {station_title} | {product_raw} | {exc}")
            continue

        results.append({
            "created_at": created_at,
            "city": city,
            "station": station,
            "fuel": fuel,
            "price": price,
            "region": region,
            "location": location,
        })

    print(
        f"[STATION] {station_title} | {city or '-'} | "
        f"{len(results)} fuel prices"
    )

    return results


def scrape_all_pages():
    all_results = []
    empty_pages = 0
    driver = create_driver()

    try:
        for offset in range(0, MAX_OFFSET + PAGE_STEP, PAGE_STEP):
            url = make_page_url(offset)

            try:
                soup = get_soup(
                    driver,
                    url,
                    first_page=(offset == 0),
                )
            except Exception as exc:
                print(f"[ERROR] Could not load {url}: {exc}")
                empty_pages += 1

                if empty_pages >= EMPTY_PAGE_LIMIT:
                    print("[STOP] Too many consecutive failed/empty pages.")
                    break

                continue

            cards = find_station_cards(soup)

            print(
                f"[PAGE] offset={offset} | "
                f"station cards={len(cards)}"
            )

            if not cards:
                empty_pages += 1

                if empty_pages >= EMPTY_PAGE_LIMIT:
                    print(
                        f"[STOP] {EMPTY_PAGE_LIMIT} consecutive pages "
                        "without station cards."
                    )
                    break

                continue

            empty_pages = 0

            page_results = []

            for card in cards:
                page_results.extend(scrape_station_card(card))

            all_results.extend(page_results)

            print(
                f"[TOTAL] rows collected so far: {len(all_results):,}"
            )

    finally:
        driver.quit()

    if not all_results:
        return pd.DataFrame(columns=COLUMNS)

    df = pd.DataFrame(all_results, columns=COLUMNS)

    # The same station/product may appear more than once while traversing
    # "last updated". Keep one copy of an identical observation.
    df = df.drop_duplicates(
        subset=[
            "created_at",
            "city",
            "station",
            "fuel",
            "price",
            "region",
            "location",
        ]
    ).reset_index(drop=True)

    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    df = scrape_all_pages()

    print("\n===== RESULT =====")
    print(df)
    print(f"\nRows: {len(df):,}")
    print(f"Stations/brands: {df['station'].nunique(dropna=True) if not df.empty else 0}")
    print(f"Cities: {df['city'].nunique(dropna=True) if not df.empty else 0}")

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\nCSV saved: {OUTPUT_CSV}")


[PAGE] https://fuelo.net/prices/last_updated?lang=bg | HTML: 756,769 chars
[PAGE] offset=0 | station cards=31
[STATION] FINES Kyosevtsi Bar & Grill | - | 0 fuel prices
[STATION] FINES Kyosevtsi Bar & Grill | - | 0 fuel prices
[STATION] FINES Camping Beglika | - | 0 fuel prices
[STATION] FINES Luda Yana | - | 0 fuel prices
[STATION] FINES Intercom Group Varna | - | 0 fuel prices
[STATION] FINES Dolen Satovcha | - | 0 fuel prices
[STATION] FINES Dexa Group Dobrich | - | 0 fuel prices
[STATION] FINES RICHHILL - Tenant & Employee Charging Only | - | 0 fuel prices
[STATION] FINES Poli Stil | - | 0 fuel prices
[STATION] FINES Retail Park Tsarevo | - | 0 fuel prices
[STATION] FINES Lavatex Kardzhali | - | 0 fuel prices
[STATION] FINES Hay Group Shumen | - | 0 fuel prices
[STATION] FINES Mr. Bricolage XOPark | - | 0 fuel prices
[STATION] FINES SAM 96 | - | 0 fuel prices
[STATION] FINES Pristan V Planinata | - | 0 fuel prices
[STATION] FINES Kranevo | - | 0 fuel prices
[STATION] FINES FiNES Omn